In [2]:
#https://pygithub.readthedocs.io/en/stable/introduction.html
from github import Github
# Authentication is defined via github.Auth
from github import Auth
import pandas as pd
import numpy as np
import json 
from datetime import datetime, date
from collections import Counter
import plotly.express as px
import time
import pickle
intrinsic_people = ["@aaronchongth","@akash-roboticist","@andreasBihlmaier","@arjo129","@audrow","@azeey","@damon-oss","@faximan","@jennuine","@koonpeng","@kscottz","@luca-della-vedova","@marcoag","@mbordignon-intrinsic","@methylDragon","@mjcarroll","@mjeronimo","@mxgrey","@nuclearsandwich-ai","@quarkytale","@scpeters","@sloretz","@tfoote","@udaya2899","@xiyuoh","@Yadunund"]
org_name = "gazebosim"
this_year = 2023
last_year = 2022

In [3]:
# Grab the access token
with open('./tokens.json',"r") as json_data:
    tokens = json.loads(json_data.read())
    json_data.close()

auth = Auth.Token(tokens["Github"])

# Public Web Github
gh = Github(auth=auth)

In [4]:
org = gh.get_organization(org_name)
repos = org.get_repos()

In [5]:
def extract_contributions(gh, repo_name, start_date, end_date):
# Get github repo level stats for two date ranges
    repo = gh.get_repo(repo_name)
    prs = repo.get_pulls(state='closed', sort='created')
    year = []
    results = {}
    count = 0
    for pr in prs:
        if start_date < pr.closed_at.date() < end_date:
            year.append(pr)
            count += 1
            
    results["repo"] = repo_name
    results["prs"] = year
    results["start_date"] = start_date    
    results["end_date"] = end_date    
    results["users"] = []
    results["handles"] = []
    results["add"] = 0
    results["del"] = 0 
    results["comments"] = 0
    results["files"] = 0
    
    for pr in year:
        results["users"].append(pr.user.name)
        results["handles"].append(pr.user.login)
        results["files"] += pr.changed_files 
        results["del"] += pr.deletions
        results["add"] += pr.additions
        results["comments"] += pr.review_comments

    results["total_prs"] = count
    results["total_users"] = len(set(results["users"]))

    return results

In [6]:
# Create a list of github repos for an org
full_repo_list = []
i = 0
has_repos = True
while has_repos:
    next_repos = repos.get_page(i)
    if len(next_repos) > 0:
        i += 1
        full_repo_list += next_repos
    else:
        has_repos = False
        
print(full_repo_list)
print(len(full_repo_list))

[Repository(full_name="gazebosim/ros_gz"), Repository(full_name="gazebosim/design"), Repository(full_name="gazebosim/gz-rviz"), Repository(full_name="gazebosim/sdf_tutorials"), Repository(full_name="gazebosim/sdformat"), Repository(full_name="gazebosim/docs"), Repository(full_name="gazebosim/ign-acropolis"), Repository(full_name="gazebosim/ign-blueprint"), Repository(full_name="gazebosim/gz-citadel"), Repository(full_name="gazebosim/gz-cmake"), Repository(full_name="gazebosim/gz-common"), Repository(full_name="gazebosim/gz-fuel-tools"), Repository(full_name="gazebosim/gz-sim"), Repository(full_name="gazebosim/ign-go1"), Repository(full_name="gazebosim/gz-gui"), Repository(full_name="gazebosim/gz-launch"), Repository(full_name="gazebosim/gz-math"), Repository(full_name="gazebosim/gz-msgs"), Repository(full_name="gazebosim/gz-physics"), Repository(full_name="gazebosim/gz-plugin"), Repository(full_name="gazebosim/gz-rendering"), Repository(full_name="gazebosim/gz-rndf"), Repository(full_n

In [7]:
this_year_start = date(this_year, 1, 1)
this_year_end = date(this_year, 12, 31)
last_year_start = date(last_year, 1, 1)
last_year_end = date(last_year, 12, 31)
full_org_results = {}
fname = 'github_stats_{0}_{1}-{2}.pkl'.format(org_name,this_year,last_year)
for repo in full_repo_list:
    print("Extracting data for {0} from {1} to {2}".format(repo.full_name,this_year_start,this_year_end))
    this_year_results = extract_contributions(gh, repo.full_name, this_year_start, this_year_end)
    print("Extracting data for {0} from {1} to {2}".format(repo.full_name,last_year_start,last_year_end))
    last_year_results = extract_contributions(gh, repo.full_name, last_year_start, last_year_end)
    full_org_results[repo.name] = {}
    full_org_results[repo.name][this_year] = this_year_results
    full_org_results[repo.name][last_year] = last_year_results
    with open(fname,"wb") as file:
        pickle.dump(full_org_results, file)
        print("Wrote: {0}".format(fname))
    print("-----------------------------")



Extracting data for gazebosim/ros_gz from 2023-01-01 to 2023-12-31
Extracting data for gazebosim/ros_gz from 2022-01-01 to 2022-12-31
Wrote: github_stats_gazebosim_2023-2022.pkl
-----------------------------
Extracting data for gazebosim/design from 2023-01-01 to 2023-12-31
Extracting data for gazebosim/design from 2022-01-01 to 2022-12-31
Wrote: github_stats_gazebosim_2023-2022.pkl
-----------------------------
Extracting data for gazebosim/gz-rviz from 2023-01-01 to 2023-12-31
Extracting data for gazebosim/gz-rviz from 2022-01-01 to 2022-12-31
Wrote: github_stats_gazebosim_2023-2022.pkl
-----------------------------
Extracting data for gazebosim/sdf_tutorials from 2023-01-01 to 2023-12-31
Extracting data for gazebosim/sdf_tutorials from 2022-01-01 to 2022-12-31
Wrote: github_stats_gazebosim_2023-2022.pkl
-----------------------------
Extracting data for gazebosim/sdformat from 2023-01-01 to 2023-12-31
Extracting data for gazebosim/sdformat from 2022-01-01 to 2022-12-31
Wrote: github_

Request GET /users/mabelzhang failed with 403: Forbidden
Setting next backoff to 679.527369s


Wrote: github_stats_gazebosim_2023-2022.pkl
-----------------------------
Extracting data for gazebosim/gz-launch from 2023-01-01 to 2023-12-31
Extracting data for gazebosim/gz-launch from 2022-01-01 to 2022-12-31
Wrote: github_stats_gazebosim_2023-2022.pkl
-----------------------------
Extracting data for gazebosim/gz-math from 2023-01-01 to 2023-12-31
Extracting data for gazebosim/gz-math from 2022-01-01 to 2022-12-31
Wrote: github_stats_gazebosim_2023-2022.pkl
-----------------------------
Extracting data for gazebosim/gz-msgs from 2023-01-01 to 2023-12-31
Extracting data for gazebosim/gz-msgs from 2022-01-01 to 2022-12-31
Wrote: github_stats_gazebosim_2023-2022.pkl
-----------------------------
Extracting data for gazebosim/gz-physics from 2023-01-01 to 2023-12-31
Extracting data for gazebosim/gz-physics from 2022-01-01 to 2022-12-31
Wrote: github_stats_gazebosim_2023-2022.pkl
-----------------------------
Extracting data for gazebosim/gz-plugin from 2023-01-01 to 2023-12-31
Extrac

Wrote: github_stats_2024.pkl
-----------------------------
Extracting data for ros2/domain_bridge from 2024-01-01 to 2024-12-31
Extracting data for ros2/domain_bridge from 2023-01-01 to 2023-12-31
Wrote: github_stats_2024.pkl
-----------------------------
Extracting data for ros2/ros_network_viz from 2024-01-01 to 2024-12-31
Extracting data for ros2/ros_network_viz from 2023-01-01 to 2023-12-31
Wrote: github_stats_2024.pkl
-----------------------------
Extracting data for ros2/orocos_kdl_vendor from 2024-01-01 to 2024-12-31
Extracting data for ros2/orocos_kdl_vendor from 2023-01-01 to 2023-12-31
Wrote: github_stats_2024.pkl
-----------------------------
Extracting data for ros2/netperf from 2024-01-01 to 2024-12-31
Extracting data for ros2/netperf from 2023-01-01 to 2023-12-31
Wrote: github_stats_2024.pkl
-----------------------------
Extracting data for ros2/rcl_content_filter_fallback from 2024-01-01 to 2024-12-31
Extracting data for ros2/rcl_content_filter_fallback from 2023-01-01 t

In [12]:
out =  None
with open(fname, 'rb') as file:
        out = pickle.load(file)      
print(len(out.keys()))
print(out.keys())

54
dict_keys(['ros_gz', 'design', 'gz-rviz', 'sdf_tutorials', 'sdformat', 'docs', 'ign-acropolis', 'ign-blueprint', 'gz-citadel', 'gz-cmake', 'gz-common', 'gz-fuel-tools', 'gz-sim', 'ign-go1', 'gz-gui', 'gz-launch', 'gz-math', 'gz-msgs', 'gz-physics', 'gz-plugin', 'gz-rendering', 'gz-rndf', 'gz-sensors', 'gz-tools', 'gz-transport', 'gazebo-classic', 'testing', 'gz-bazel', 'ign-dome', 'gz-edifice', 'gz-utils', '.github', 'gz-fortress', 'fortress_demo', 'gz-garden', 'gz-omni', 'gz-omni-meta', 'ci-test', 'gz-mujoco', 'gz-usd', 'gz_pkg_create', 'garden_demo', 'garden-tutorial-party', 'ros_gz_project_template', 'gz-chrono', 'gz-test', 'gz-harmonic', 'harmonic_demo', 'gazebo_test_cases', 'gz-ionic', 'ionic_demo', 'rules_gazebo', 'connect-samples', 'gz-jetty'])


In [13]:
# Do full org aggregation
to_agg = ["users","add","del","files"]

full_results = {}
full_results[this_year] = {}
full_results[last_year] = {}

first = True
for key in full_org_results.keys():
    if first:
        full_results[this_year] = {k: full_org_results[key][this_year][k] for k in to_agg}
        full_results[last_year] = {k: full_org_results[key][last_year][k] for k in to_agg}
        full_results[this_year]["prs"] = len(full_org_results[key][this_year]["prs"])
        full_results[last_year]["prs"] = len(full_org_results[key][last_year]["prs"])
        first = False
    else:        
        for a in to_agg:
            full_results[this_year][a] += full_org_results[key][this_year][a]
            full_results[last_year][a] += full_org_results[key][last_year][a]
            full_results[this_year]["prs"] += len(full_org_results[key][this_year]["prs"])
            full_results[last_year]["prs"] += len(full_org_results[key][last_year]["prs"])

full_results[last_year]["contributors"] = set(full_results[last_year]["users"])
full_results[last_year]["users"] = len(set(full_results[last_year]["users"]))

full_results[this_year]["contributors"] = set(full_results[this_year]["users"])
full_results[this_year]["users"] = len(set(full_results[this_year]["users"]))


print("Results for {0} ==> {1}".format(last_year,this_year))
print("-------------------------")

temp_this = {}
temp_last = {}
temp_change = {}

for k in full_results[this_year].keys():
    if k == "contributors":
        continue
    change = -100*(full_results[last_year][k]-full_results[this_year][k])/full_results[last_year][k]
    temp_this[k] =  full_results[this_year][k]
    temp_last[k] =  full_results[last_year][k]
    temp_change[k] = change
    print("{0:6s}| {1} : {2:<6} | {3} : {4:<6} | {5:4.2f}%".format(k,last_year,full_results[last_year][k],this_year,full_results[this_year][k],change))
print(full_results[this_year]["contributors"])

summary_results = pd.DataFrame(data=[temp_this,temp_last,temp_change])
summary_results.to_csv("{0}-{1}-{2}-GithubContribsSummary.csv".format(org_name,last_year,this_year))

Results for 2022 ==> 2023
-------------------------
users | 2022 : 98     | 2023 : 85     | -13.27%
add   | 2022 : 2427639 | 2023 : 5825381 | 139.96%
del   | 2022 : 892250 | 2023 : 818013 | -8.32%
files | 2022 : 40166  | 2023 : 16526  | -58.86%
prs   | 2022 : 10566  | 2023 : 6109   | -42.18%
{'Joan Aguilar Mayans', 'Martin Pecka', 'Zaidhaan', None, 'Louise Poubel', 'Konstantinos Chatzilygeroudis', 'methylDragon', 'Tejal Ashwini Barnwal', 'Øystein Sture', 'Levi Armstrong', 'Vít Ondruch', 'Henrique Barros Oliveira', 'Dharini Dutia', 'Aaron Chong', 'Ash Babu', 'Angelo Elias Dal Zotto', 'Terry Welsh', 'Ryan', 'Will Stott', 'Anton Bredenbeck', 'El Jawad Alaa', 'Nate Koenig', 'Matthew LeMay', 'Carlos Agüero', 'Katherine Scott', 'Connor Taylor', 'Scott K Logan', 'Andrej Orsula', 'Anas Aamoum', 'Silvio Traversaro', 'Ian Chen', 'Arjo Chakravarty', 'Ivan Santiago Paunovic', 'Eloy Briceno', 'Jenn Nguyen', 'Tully Foote', 'Valentina Vasco', 'Michael Carroll', 'Samaahita Belavadi', 'Jose Luis Rivero

In [14]:
target = "add"
agg_result = []
for key in full_org_results.keys():
    a = full_org_results[key][this_year][target]
    b = full_org_results[key][last_year][target]
    delta = 0.00
    if b > 0:
        delta = (-100.0*(b-a)/b)
    temp = {}
    temp["name"] = key
    temp[this_year] = a
    temp[last_year] = b
    temp["change"] = delta
    agg_result.append(temp)
    
newlist = sorted(agg_result, key=lambda d: d[this_year])
newlist.reverse()
print("Results for '{0}' across {1} org".format(target,org))
print("-------------------------------------------------------------------")
for i in newlist:  
    print("{0:24s}| 2023: {1:<8} | 2024: {2:<8} | delta: {3:4.2f}%".format(i["name"][:24],
                                                                           i[last_year],
                                                                           i[this_year],
                                                                           i["change"]))
    
df = pd.DataFrame(data=newlist)
df.to_csv("{0}-{1}-{2}-NewLines.csv".format(org_name,last_year,this_year))

Results for 'add' across Organization(login="gazebosim") org
-------------------------------------------------------------------
harmonic_demo           | 2023: 0        | 2024: 4817080  | delta: 0.00%
gz-sim                  | 2023: 250983   | 2024: 649529   | delta: 158.79%
gz-rendering            | 2023: 173673   | 2024: 55290    | delta: -68.16%
gz-transport            | 2023: 53843    | 2024: 53473    | delta: -0.69%
sdformat                | 2023: 128889   | 2024: 47551    | delta: -63.11%
gz-physics              | 2023: 59093    | 2024: 31659    | delta: -46.43%
gz-gui                  | 2023: 50421    | 2024: 30759    | delta: -39.00%
gz-common               | 2023: 78294    | 2024: 24620    | delta: -68.55%
docs                    | 2023: 24133    | 2024: 24552    | delta: 1.74%
gz-msgs                 | 2023: 48901    | 2024: 17953    | delta: -63.29%
gz-sensors              | 2023: 42648    | 2024: 14878    | delta: -65.11%
gz-bazel                | 2023: 7        | 2024: 11

In [19]:
target = "del"
agg_result = []
for key in full_org_results.keys():
    a = full_org_results[key][this_year][target]
    b = full_org_results[key][last_year][target]
    delta = 0.00
    if b > 0:
        delta = (-100.0*(b-a)/b)
    temp = {}
    temp["name"] = key
    temp[this_year] = a
    temp[last_year] = b
    temp["change"] = delta
    agg_result.append(temp)
    
newlist = sorted(agg_result, key=lambda d: d[this_year])
newlist.reverse()
print("Results for '{0}' across {1} org".format(target,org))
print("-------------------------------------------------------------------")
for i in newlist:  
    print("{0:24s}| 2023: {1:<8} | 2024: {2:<8} | delta: {3:4.2f}%".format(i["name"][:24],
                                                                           i[last_year],
                                                                           i[this_year],
                                                                           i["change"]))
    
df = pd.DataFrame(data=newlist)
df.to_csv("{0}-{1}-{2}-NewLines.csv".format(org_name,last_year,this_year))

Results for 'del' across Organization(login="gazebosim") org
-------------------------------------------------------------------
harmonic_demo           | 2023: 0        | 2024: 564278   | delta: 0.00%
gz-sim                  | 2023: 91957    | 2024: 50516    | delta: -45.07%
gz-rendering            | 2023: 102670   | 2024: 47089    | delta: -54.14%
gz-transport            | 2023: 51250    | 2024: 28064    | delta: -45.24%
gz-gui                  | 2023: 42711    | 2024: 23120    | delta: -45.87%
gz-msgs                 | 2023: 39064    | 2024: 17452    | delta: -55.32%
gz-physics              | 2023: 51664    | 2024: 16964    | delta: -67.16%
gz-common               | 2023: 66590    | 2024: 10848    | delta: -83.71%
sdformat                | 2023: 88562    | 2024: 8943     | delta: -89.90%
ros_gz                  | 2023: 27899    | 2024: 8109     | delta: -70.93%
gz-sensors              | 2023: 35161    | 2024: 7848     | delta: -77.68%
gz-math                 | 2023: 69758    | 2024:

In [18]:
target = "prs"

agg_result = []
for key in full_org_results.keys():
    a = len(full_org_results[key][this_year][target])
    b = len(full_org_results[key][last_year][target])
    delta = 0.00
    if b > 0:
        delta = (-100.0*(b-a)/b)
    temp = {}
    temp["name"] = key
    temp[this_year] = a
    temp[last_year] = b
    temp["change"] = delta
    agg_result.append(temp)
    
newlist = sorted(agg_result, key=lambda d: d[this_year])
newlist.reverse()
print("PR count by year")
print("Results for '{0}' across ROS 2 org".format(target))
print("-------------------------------------------------------------------")
for i in newlist:  
    print("{0:24s}| 2023: {1:<8} | 2024: {2:<8} | delta: {3:4.2f}%".format(i["name"][:24],
                                                                           i[last_year],
                                                                           i[this_year],
                                                                           i["change"]))
df = pd.DataFrame(data=newlist)
df.to_csv("{0}-{1}-{2}-PRS.csv".format(org,last_year,this_year))

PR count by year
Results for 'prs' across ROS 2 org
-------------------------------------------------------------------
gz-sim                  | 2023: 442      | 2024: 291      | delta: -34.16%
sdformat                | 2023: 342      | 2024: 128      | delta: -62.57%
gz-rendering            | 2023: 228      | 2024: 122      | delta: -46.49%
gz-physics              | 2023: 126      | 2024: 100      | delta: -20.63%
ros_gz                  | 2023: 102      | 2024: 85       | delta: -16.67%
gz-sensors              | 2023: 106      | 2024: 82       | delta: -22.64%
gz-msgs                 | 2023: 110      | 2024: 82       | delta: -25.45%
gz-gui                  | 2023: 145      | 2024: 77       | delta: -46.90%
gz-transport            | 2023: 82       | 2024: 75       | delta: -8.54%
gz-fuel-tools           | 2023: 86       | 2024: 72       | delta: -16.28%
gz-common               | 2023: 173      | 2024: 72       | delta: -58.38%
docs                    | 2023: 90       | 2024: 72     